In [1]:
import torch
from torchvision import datasets, transforms
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

import sys
import os
sys.path.append(os.path.abspath("../.."))

from trainingGPU import train_model
from testingGPU import model_testing
from src.CNN_implementation import CNN

In [2]:
# We will need to convert image to tensors
# here we define a transform
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

transform = transforms.ToTensor()

cuda


In [3]:
# load the dataset
# now each image mnist_trainset[i][0] is a tensor of shape [1, 28, 28] (1 channel, 28×28 pixels)
mnist_trainset=datasets.MNIST(root = './data', train = True, download = True, transform = transform)
mnist_testset=datasets.MNIST(root = './data', train = False, download = True, transform = transform)

In [4]:
#len(mnist_trainset) len(mnist_testset)

In [5]:
image, label = mnist_trainset[0]

print(type(image))
print(image.shape)
print(label)

<class 'torch.Tensor'>
torch.Size([1, 28, 28])
5


In [6]:
#Create CNN
x = image.unsqueeze(0).to(device)
model = CNN().to(device)
out = model(x)
print(out.shape)

torch.Size([1, 10])


In [7]:
# Training
train_loader = DataLoader(mnist_trainset, batch_size = 64, shuffle = True)
train_model(train_loader, model, device)

# Test the accuracy on test data
test_loader = DataLoader(mnist_testset, batch_size = 64, shuffle = False)# loading

model_testing(test_loader, model, device)

Epoch 1/3, Average Loss: 0.8700
Epoch 2/3, Average Loss: 0.2889
Epoch 3/3, Average Loss: 0.2038
Test accuracy: 95.51%


In [14]:
"""
# Train centralised model

# Load the data
# Instead of 1 image at a time, we work in batches
train_loader = DataLoader(mnist_trainset, batch_size = 64, shuffle = True)

#for images, labels in train_loader:
#    print(images.shape)    # torch.Size([64, 1, 28, 28])
#    print(labels.shape)    # torch.Size([64])
#    break

# Loss function and optimiser
# We will use CrossEntropy for loss and Adam for optimisation 
criterion = nn.CrossEntropyLoss()
optimiser = optim.Adam(model.parameters(), lr = 0.001) # here the learning rate is 0.001

# train one batch
images, labels = next(iter(train_loader))
outputs = model(images)
loss = criterion(outputs, labels) # compute the loss
print('Loss before backward propagation: ',loss.item())

# Backward propagation
optimiser.zero_grad() # clear previous gradients
loss.backward() # compute gradients
# update the weights
optimiser.step()
# check loss again
outputs_new = model(images)
loss_new = criterion(outputs_new, labels)
print('Loss after one backward propagation step: ',loss_new.item())
"""


"\n# Train centralised model\n\n# Load the data\n# Instead of 1 image at a time, we work in batches\ntrain_loader = DataLoader(mnist_trainset, batch_size = 64, shuffle = True)\n\n#for images, labels in train_loader:\n#    print(images.shape)    # torch.Size([64, 1, 28, 28])\n#    print(labels.shape)    # torch.Size([64])\n#    break\n\n# Loss function and optimiser\n# We will use CrossEntropy for loss and Adam for optimisation \ncriterion = nn.CrossEntropyLoss()\noptimiser = optim.Adam(model.parameters(), lr = 0.001) # here the learning rate is 0.001\n\n# train one batch\nimages, labels = next(iter(train_loader))\noutputs = model(images)\nloss = criterion(outputs, labels) # compute the loss\nprint('Loss before backward propagation: ',loss.item())\n\n# Backward propagation\noptimiser.zero_grad() # clear previous gradients\nloss.backward() # compute gradients\n# update the weights\noptimiser.step()\n# check loss again\noutputs_new = model(images)\nloss_new = criterion(outputs_new, labels